In [1]:
print("Hello world")

Hello world


In [12]:
import cv2
import mediapipe as mp
import numpy as np
import time

def run_emotion_recognition():
    cap = cv2.VideoCapture(0)
    face_mesh = mp.solutions.face_mesh.FaceMesh(static_image_mode=False, max_num_faces=1)
    last_print = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        h, w = frame.shape[:2]
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = face_mesh.process(rgb)
        if results.multi_face_landmarks:
            lm = results.multi_face_landmarks[0].landmark
            mar_num = np.linalg.norm([lm[13].x*w - lm[14].x*w, lm[13].y*h - lm[14].y*h])
            # normalize by face height (forehead to chin)
            face_top = lm[10]
            face_bottom = lm[152]
            face_height = np.linalg.norm([
                (face_top.x - face_bottom.x) * w,
                (face_top.y - face_bottom.y) * h
            ])
            mar = mar_num / max(face_height, 1e-6)
            if mar > 0.085:
                emotion = "surprised"
            elif mar > 0.055:
                emotion = "happy"
            else:
                emotion = "neutral"
            t = time.time()
            if t - last_print > 2:
                print("MAR:", round(mar, 4), "Emotion:", emotion)
                last_print = t
        if cv2.waitKey(1) & 0xFF == 27: break
    cap.release()
    cv2.destroyAllWindows()


In [1]:
# Alternative: Use DeepFace, a state-of-the-art facial attribute analysis library
# DeepFace supports emotion recognition using pre-trained deep learning models.
from deepface import DeepFace
import cv2
import sys

def run_emotion_recognition_live():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open camera")
        return

    try:
        # Environment diagnostics to catch kernel/env mismatches
        import deepface
        print("Python:", sys.executable)
        print("DeepFace version:", getattr(deepface, "__version__", "unknown"))
        # Force-reload modeling module in case the kernel cached an older DeepFace
        import importlib
        from deepface.modules import modeling as _df_modeling
        importlib.reload(_df_modeling)
        print("deepface.modules.modeling path:", getattr(_df_modeling, "__file__", None))
        print("has build_model:", hasattr(_df_modeling, "build_model"))
    except Exception as e:
        print("Error during DeepFace diagnostics:", e)
        cap.release()
        return

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame from webcam")
                break

            # DeepFace expects BGR image array directly for analyze (img_path=None)
            try:
                results = DeepFace.analyze(
                    frame,
                    actions=['emotion'],
                    enforce_detection=True,
                    detector_backend='opencv'
                )
                if isinstance(results, list) and results:
                    results = results[0]
                emotion = results.get('dominant_emotion') if isinstance(results, dict) else None
            except Exception as e:
                emotion = None
                print("DeepFace error:", e)
            
            show_frame = frame.copy()
            if emotion:
                cv2.putText(show_frame, f"Emotion: {emotion}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 2)
            cv2.imshow("Emotion Recognition (Live)", show_frame)

            if cv2.waitKey(1) & 0xFF == 27:
                break  # ESC to exit

    except Exception as e:
        print("Error during live emotion recognition:", e)
    finally:
        cap.release()
        cv2.destroyAllWindows()



In [2]:
run_emotion_recognition_live()

Python: d:\projects\Adaptive_UI\final_project\.venv\Scripts\python.exe
DeepFace version: 0.0.95
deepface.modules.modeling path: d:\projects\Adaptive_UI\final_project\.venv\Lib\site-packages\deepface\modules\modeling.py
has build_model: True
DeepFace error: Face could not be detected in numpy array.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
DeepFace error: Face could not be detected in numpy array.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
DeepFace error: Face could not be detected in numpy array.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
DeepFace error: Face could not be detected in numpy array.Please confirm that the picture is a face photo or consider to set enforce_detection param to False.
DeepFace error: Face could not be detected in numpy array.Please confirm that the picture is a face photo or consider to se

In [8]:
# Improved DeepFace emotion recognition with better face detection
from deepface import DeepFace
import cv2
import sys
import numpy as np

def run_emotion_recognition_improved():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open camera")
        return

    # Set camera properties for better quality
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    cap.set(cv2.CAP_PROP_FPS, 30)

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Failed to grab frame from webcam")
                break

            # Preprocess the frame for better face detection
            # Convert to RGB (DeepFace expects RGB)
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            # Resize frame if it's too large (helps with detection)
            height, width = rgb_frame.shape[:2]
            if width > 800:
                scale = 800 / width
                new_width = int(width * scale)
                new_height = int(height * scale)
                rgb_frame = cv2.resize(rgb_frame, (new_width, new_height))
            
            # Try multiple detection backends and settings
            emotion = None
            detection_success = False
            
            # Try different backends in order of preference
            backends = ['opencv', 'mtcnn', 'retinaface', 'ssd']
            
            for backend in backends:
                try:
                    results = DeepFace.analyze(
                        rgb_frame,
                        actions=['emotion'],
                        enforce_detection=True,
                        detector_backend=backend,
                        silent=True  # Reduce verbose output
                    )
                    
                    if isinstance(results, list) and results:
                        results = results[0]
                    
                    if isinstance(results, dict) and 'dominant_emotion' in results:
                        emotion = results['dominant_emotion']
                        detection_success = True
                        break
                        
                except Exception as e:
                    # If this backend fails, try the next one
                    continue
            
            # If all backends failed, try with enforce_detection=False
            if not detection_success:
                try:
                    results = DeepFace.analyze(
                        rgb_frame,
                        actions=['emotion'],
                        enforce_detection=False,  # Don't enforce face detection
                        detector_backend='opencv',
                        silent=True
                    )
                    
                    if isinstance(results, list) and results:
                        results = results[0]
                    
                    if isinstance(results, dict) and 'dominant_emotion' in results:
                        emotion = results['dominant_emotion']
                        detection_success = True
                        
                except Exception as e:
                    pass
            
            # Display the frame with emotion result
            display_frame = frame.copy()
            
            if emotion:
                cv2.putText(display_frame, f"Emotion: {emotion}", (10, 40), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
            else:
                cv2.putText(display_frame, "No face detected", (10, 40), 
                           cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
                cv2.putText(display_frame, "Ensure good lighting", (10, 80), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
            cv2.imshow("Improved Emotion Recognition", display_frame)

            if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
                break

    except Exception as e:
        print("Error during live emotion recognition:", e)
    finally:
        cap.release()
        cv2.destroyAllWindows()


In [ ]:
# # Alternative approach: Use OpenCV face detection + DeepFace emotion analysis
# import cv2
# import numpy as np
# from deepface import DeepFace

# def run_emotion_recognition_with_opencv_detection():
#     cap = cv2.VideoCapture(0)
#     if not cap.isOpened():
#         print("Cannot open camera")
#         return

#     # Load OpenCV's face cascade classifier
#     face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    
#     # Set camera properties
#     cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
#     cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
#     cap.set(cv2.CAP_PROP_FPS, 30)

#     try:
#         while True:
#             ret, frame = cap.read()
#             if not ret:
#                 print("Failed to grab frame from webcam")
#                 break

#             # Convert to grayscale for face detection
#             gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
#             # Detect faces using OpenCV
#             faces = face_cascade.detectMultiScale(
#                 gray, 
#                 scaleFactor=1.1, 
#                 minNeighbors=5, 
#                 minSize=(30, 30)
#             )
            
#             emotion = None
#             if len(faces) > 0:
#                 # Get the largest face (most likely the main subject)
#                 largest_face = max(faces, key=lambda x: x[2] * x[3])
#                 x, y, w, h = largest_face
                
#                 # Extract face region
#                 face_roi = frame[y:y+h, x:x+w]
                
#                 # Convert to RGB for DeepFace
#                 face_rgb = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)
                
#                 try:
#                     # Analyze emotion on the cropped face
#                     results = DeepFace.analyze(
#                         face_rgb,
#                         actions=['emotion'],
#                         enforce_detection=False,  # We already detected the face
#                         detector_backend='opencv',
#                         silent=True
#                     )
                    
#                     if isinstance(results, list) and results:
#                         results = results[0]
                    
#                     if isinstance(results, dict) and 'dominant_emotion' in results:
#                         emotion = results['dominant_emotion']
                        
#                 except Exception as e:
#                     # If DeepFace fails, we'll just show "Face detected" without emotion
#                     pass
                
#                 # Draw rectangle around detected face
#                 cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
            
#             # Display emotion result
#             if emotion:
#                 cv2.putText(frame, f"Emotion: {emotion}", (10, 40), 
#                            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 0), 2)
#             elif len(faces) > 0:
#                 cv2.putText(frame, "Face detected - analyzing...", (10, 40), 
#                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 0), 2)
#             else:
#                 cv2.putText(frame, "No face detected", (10, 40), 
#                            cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)
#                 cv2.putText(frame, "Position your face in the camera", (10, 80), 
#                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
            
#             cv2.imshow("Emotion Recognition (OpenCV + DeepFace)", frame)

#             if cv2.waitKey(1) & 0xFF == 27:  # ESC to exit
#                 break

#     except Exception as e:
#         print("Error during emotion recognition:", e)
#     finally:
#         cap.release()
#         cv2.destroyAllWindows()


In [ ]:
# # Test the improved emotion recognition
# # print("Testing improved emotion recognition...")
# print("Choose one of the following methods:")
# print("1. run_emotion_recognition_improved() - Multiple DeepFace backends")
# print("2. run_emotion_recognition_with_opencv_detection() - OpenCV + DeepFace")
# print("\nRecommended: Try method 2 first as it's more reliable for face detection")


Testing improved emotion recognition...
Choose one of the following methods:
1. run_emotion_recognition_improved() - Multiple DeepFace backends
2. run_emotion_recognition_with_opencv_detection() - OpenCV + DeepFace

Recommended: Try method 2 first as it's more reliable for face detection


In [ ]:
# run_emotion_recognition_with_opencv_detection()

In [9]:
run_emotion_recognition_improved()

25-10-18 19:09:32 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: C:\Users\ACER\.deepface\weights\retinaface.h5
100%|██████████| 119M/119M [00:23<00:00, 5.01MB/s] 
